# Lezione 5 — Pratica: classificatore binario quantistico

Hai già addestrato il classificatore binario dell'esempio usando l'operatore di lettura `cirq.X(qubit)`. In questa pratica cambi la base di misura e osservi come ciò influisce sull'apprendimento del modello.

## Traccia

Cambia l'operatore di lettura da `cirq.X(qubit)` a `cirq.Z(qubit)` e riaddestra il modello da zero. Confronta la curva di accuratezza di validazione con quella dell'esempio originale.

Se il modello non impara altrettanto bene, rifletti sul perché: ricorda che i dati sono stati codificati con una rotazione sull'asse X (`cirq.X(qubit)**angolo`), quindi la base di misura più informativa non deve per forza coincidere con la base di codifica.

**Suggerimenti**

- Riutilizza `generar_datos`, `construir_circuito` e `precision_hinge` così come sono
- Cambia solo la lista `operador_lectura`
- Crea di nuovo il modello, il layer `PQC` e il `compile` (un modello nuovo, non riaddestrare quello precedente)
- Salva la cronologia in una variabile diversa, così da poter confrontare le due curve nello stesso grafico

In [ ]:
import tensorflow_quantum as tfq
import tensorflow as tf
import cirq
import sympy
import random
import numpy as np
import matplotlib.pyplot as plt


def generar_datos(n_entrenamiento, n_prueba):
    """Genera ángulos cercanos a 0 (etiqueta +1) o cercanos a 1 (etiqueta -1)."""
    qubit = cirq.GridQubit(0, 0)

    def lote(n):
        circuitos, etiquetas = [], []
        for _ in range(n):
            angulo = random.uniform(0, 0.1) if random.random() < 0.5 else random.uniform(0.9, 1)
            circuito = cirq.Circuit()
            circuito.append(cirq.X(qubit) ** angulo)
            circuitos.append(circuito)
            etiquetas.append(1 if angulo < 0.5 else -1)
        return tfq.convert_to_tensor(circuitos), np.array(etiquetas)

    entrenamiento, etiquetas_entrenamiento = lote(n_entrenamiento)
    prueba, etiquetas_prueba = lote(n_prueba)
    return entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba


def construir_circuito(qubit):
    x, y, z = sympy.symbols("rotacion_x rotacion_y rotacion_z")
    circuito = cirq.Circuit()
    circuito.append(cirq.rx(x).on(qubit))
    circuito.append(cirq.ry(y).on(qubit))
    circuito.append(cirq.rz(z).on(qubit))
    return circuito


def precision_hinge(etiqueta_real, prediccion):
    etiqueta_real = tf.squeeze(etiqueta_real) > 0.0
    prediccion = tf.squeeze(prediccion) > 0.0
    return tf.reduce_mean(tf.cast(etiqueta_real == prediccion, tf.float32))

In [ ]:
# Reutiliza los mismos datos para que la comparación entre las dos bases de medida sea justa
entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba = generar_datos(1000, 100)

qubit = cirq.GridQubit(0, 0)

# TODO: cambia el operador de lectura de cirq.X(qubit) a cirq.Z(qubit)
operador_lectura = [cirq.X(qubit)]

entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)

# TODO: construye la capa tfq.layers.PQC con construir_circuito(qubit), operador_lectura,
# repetitions=32, differentiator=tfq.differentiators.ParameterShift() e
# initializer=tf.keras.initializers.Zeros, aplicada sobre 'entradas'
capa_cuantica = None

# TODO: crea el modelo de Keras (tf.keras.models.Model) con inputs=entradas y outputs=capa_cuantica
modelo_z = None

# TODO: compila el modelo con Adam(learning_rate=0.001), loss=tf.keras.losses.hinge
# y metrics=[precision_hinge]


In [ ]:
# TODO: entrena modelo_z con .fit(...) durante 64 épocas, batch_size=32 y
# validation_data=(prueba, etiquetas_prueba); guarda el resultado en historial_z
historial_z = None

# TODO: dibuja historial_z.history["precision_hinge"] y historial_z.history["val_precision_hinge"]
# junto con un título y leyenda que indiquen que este modelo usa cirq.Z(qubit) como lectura


<details>
<summary>💡 Mostra soluzione</summary>

```python
entrenamiento, etiquetas_entrenamiento, prueba, etiquetas_prueba = generar_datos(1000, 100)

qubit = cirq.GridQubit(0, 0)

# Modifica chiave: la lettura ora avviene nella base Z invece che nella base X
operador_lectura = [cirq.Z(qubit)]

entradas = tf.keras.Input(shape=(), dtype=tf.dtypes.string)

capa_cuantica = tfq.layers.PQC(
    construir_circuito(qubit),
    operador_lectura,
    repetitions=32,
    differentiator=tfq.differentiators.ParameterShift(),
    initializer=tf.keras.initializers.Zeros,
)(entradas)

modelo_z = tf.keras.models.Model(inputs=entradas, outputs=capa_cuantica)
modelo_z.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.hinge,
    metrics=[precision_hinge],
)

historial_z = modelo_z.fit(
    entrenamiento,
    etiquetas_entrenamiento,
    epochs=64,
    batch_size=32,
    validation_data=(prueba, etiquetas_prueba),
)

plt.plot(historial_z.history["precision_hinge"], label="Addestramento (lettura Z)")
plt.plot(historial_z.history["val_precision_hinge"], label="Validazione (lettura Z)")
plt.legend()
plt.title("Accuratezza del classificatore con lettura in base Z")
plt.xlabel("Epoche")
plt.ylabel("Accuratezza")
plt.show()

# Ragionamento: i dati vengono codificati con cirq.X(qubit)**angolo, una rotazione attorno
# all'asse X. Questa rotazione sposta lo stato nel piano Y-Z della sfera di Bloch,
# quindi l'informazione sull'angolo è codificata soprattutto nelle componenti
# Y e Z dello stato, non nella componente X. Misurare con cirq.Z(qubit) può catturare
# questa informazione peggio (o meglio, a seconda dei casi) rispetto a misurare con
# cirq.X(qubit), perché la base di misura non coincide con la base di codifica.
# Se la curva di accuratezza con lettura Z converge più lentamente o si blocca sotto
# il 100%, è un segnale che il circuito addestrabile (rx, ry, rz) deve compensare
# questo disallineamento regolando le sue tre rotazioni, il che può richiedere più
# epoche o non convergere altrettanto bene.
```

</details>